In [1]:
import numpy as np
import pandas as pd
import zsx_some_tools as st
import zsx_facile_heatmap as fh

import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
from typing import Union, Optional

In [2]:
def iterative_sigma_clipping(data, k=2, max_iter=100, tol=1e-4, tail='right', return_rounds=False):
    """
    ISC
    
    参数:
        data (array): 输入数据（近似正态分布，含异常值）
        k (float): 截尾系数（默认2，即 μ ± kσ）
        max_iter (int): 最大迭代次数
        tol (float): 收敛阈值（参数变化小于此值停止）
        tail (str): 截尾方向，'left'或'right'（默认右侧）
    
    返回:
        mu (float): 截尾后分布的均值
        sigma (float): 截尾后分布的标准差
        threshold (float): 最终截尾阈值
    """
    current_data = data.copy()
    mu_prev, sigma_prev = np.mean(data), np.std(data)
    
    for rounds in range(max_iter):
        # 根据方向计算阈值
        if tail == 'right':
            threshold = mu_prev + k * sigma_prev
            truncated_data = current_data[current_data <= threshold]
        elif tail == 'left':
            threshold = mu_prev - k * sigma_prev
            truncated_data = current_data[current_data >= threshold]
        else:
            raise ValueError("tail should be'left' or 'right'")
        
        # 更新参数
        mu_new, sigma_new = np.mean(truncated_data), np.std(truncated_data)
        
        # 检查收敛
        if np.abs(mu_new - mu_prev) < tol and np.abs(sigma_new - sigma_prev) < tol:
            break
            
        mu_prev, sigma_prev = mu_new, sigma_new
        current_data = truncated_data
    
    if not return_rounds:
        return mu_new, sigma_new, threshold
    else:
        return mu_new, sigma_new, threshold, rounds

In [3]:
def get_ISC_thres(value, tail='right', k=2, return_rounds=False):
    value.sort()
    value = np.array(value)

    max_iter = len(value) * 0.3
    if not return_rounds:
        mu, sigma, threshold = iterative_sigma_clipping(np.array(value), k=k, max_iter=100, tol=1e-4, tail=tail)
        return threshold
    
    else:
        mu, sigma, threshold, rounds = iterative_sigma_clipping(np.array(value), k=k, max_iter=100, tol=1e-4, tail=tail, 
                                                               return_rounds=return_rounds)
        return threshold, rounds

In [4]:
def rank_rows_descending_with_ties(df, keep_first_col=True):
    """
    对DataFrame的每一行进行降序排序编号，相同值获得相同排名
    正确处理NaN值，不会尝试将NaN转换为整数
    
    参数:
    df: pandas DataFrame, 输入数据
    keep_first_col: bool, 是否保持第一列不变（默认为True）
    
    返回:
    pandas DataFrame, 转换后的数据
    """
    result_df = df.copy()
    
    if keep_first_col:
        first_col = df.iloc[:, 0].copy()
        data_to_rank = df.iloc[:, 1:]
    else:
        data_to_rank = df.copy()
    
    # 创建一个空的DataFrame来存储排名结果，保持原始数据类型
    ranked_data = pd.DataFrame(index=data_to_rank.index, columns=data_to_rank.columns)
    
    # 对每一行进行处理
    for idx, row in data_to_rank.iterrows():
        # 创建一个掩码来标识非NaN值
        non_nan_mask = row.notna()
        
        if non_nan_mask.any():  # 如果该行有非NaN值
            # 只对非NaN值进行排名
            non_nan_values = row[non_nan_mask]
            
            # 进行降序排名（相同值获得相同排名）
            ranks = non_nan_values.rank(method='min', ascending=False)
            
            # 将排名结果填回到相应位置
            ranked_row = row.copy()
            ranked_row[non_nan_mask] = ranks
            ranked_data.loc[idx] = ranked_row
        else:
            # 如果整行都是NaN，保持原样
            ranked_data.loc[idx] = row
    
    if keep_first_col:
        # 组合结果，保持第一列不变
        result_df = pd.concat([first_col, ranked_data], axis=1)
        result_df.columns = df.columns
    else:
        result_df = ranked_data
    
    return result_df

In [5]:
def rank_rows_descending_with_ties_int(df, keep_first_col=True):
    """
    返回整数排名，NaN值保持为NaN
    """
    result = rank_rows_descending_with_ties(df, keep_first_col)
    
    # 将非NaN值转换为整数
    if keep_first_col:
        cols_to_convert = result.columns[1:]
    else:
        cols_to_convert = result.columns
    
    for col in cols_to_convert:
        # 只转换非NaN值
        non_nan_mask = result[col].notna()
        result.loc[non_nan_mask, col] = result.loc[non_nan_mask, col].astype(int)
    
    return result

# color

In [6]:
DC = st.DecentColor()
from matplotlib.colors import LinearSegmentedColormap

# 定义颜色列表（可以是 HEX 或 RGB）
colors = [DC.c_blue, "white", DC.c_red]
custom_cmap = LinearSegmentedColormap.from_list('custom_cmap', colors)

# My predict data¶

In [7]:
predict_path = 'D:/data/ZSX/DPF/data/DCMS_result/'
predict_files = st.listdir(predict_path, exception='result_from_code')

# Database

In [8]:
# adrb2 beta-lactamase env ha_h1 ha_h3 infa mapk1 p53 pafa

In [9]:
path = 'D:/论文/张森欣/language_model/nbt_data/data/dms/'
save_path = 'D:/data/ZSX/DPF/data/ana/figure/'
files = st.listdir(path)
st.mkdir(save_path)

add_path = 'D:/论文/张森欣/language_model/nbt_data/data/dms_addition/'

D:/data/ZSX/Protein_meta_learning/data/ana/figure  - Directory already exists


In [10]:
file_name_dict = {'dms_adrb2.csv': 'result_ADRB2.txt', 
                  'dms_bla.csv': 'result_beta-lactamase.txt', 
                  'dms_env.csv': 'result_Env.txt', 
                  'dms_ha_h1.csv': 'result_HA_H1_almost_same.txt',
                  'dms_ha_h3.csv': 'result_HA_H3_almost_same.txt', 
                  'dms_infa.csv': 'result_infA.txt', 
                  'dms_mapk1.csv': 'result_MAPK1.txt', 
                  'dms_p53.csv': 'result_P53.txt', 
                  'dms_pafa.csv': 'result_PafA.txt'}
protein_name_dict = {}
for key, value in file_name_dict.items():
    protein_name = value.split('result_')[-1].split('.txt')[0].split('_almost')[0]
    protein_name_dict[protein_name] = key

protein_list = ['ADRB2', 'beta-lactamase', 'Env', 'HA_H1', 'HA_H3', 'infA', 'MAPK1', 'P53', 'PafA']
property_list = ['catalytic_activity', 'DSD_binding', 'folding_stability',
                 'Immunogenicity', 'Photoactivity', 'pH_stability', 'redox_activity',
                 'SSD_binding', 'thermal_stability', 'SSD_thermal']

In [11]:
k2p_list = [1.645, 1.96, 2.576, 3.291]
alpha_list = [0.1, 0.05, 0.01, 0.001]

result_all = []
logging = []
for file in files:
    data = st.read_file(path + file, index_col=0)
    pre_name = file_name_dict[file]
    protein_name = pre_name.split('result_')[-1].split('.txt')[0].split('_almost')[0]
    target_columns = [i for i in data.columns if 'DMS' in i]
    target_number = len(target_columns)
    
    data_pre = st.read_file(predict_path + pre_name, index_col=0)
    pre_col_use = [i for i in data_pre.columns if '.1' not in i]
    data_pre = data_pre.loc[:, pre_col_use]

    overlap = list(set(data_pre.index) & set(data.index))
    overlap.sort()
    if len(overlap) < data.shape[0] * 0.5:
        if 'PafA' not in pre_name:
            logging += [[protein_name, '', '', 'overlap not enough']]
            continue

    data_all = pd.concat([data_pre, pd.DataFrame(None, columns=['--']), data], axis=1)
    
    ### corr ###
    target_data = pd.concat([data_all.iloc[:, :11], data_all.iloc[:, -target_number:]], axis=1)  # .dropna()
    corr_da = target_data.corr().iloc[-target_number:, :]
    
    ### scatter ###
    for i in range(10):
        col1 = target_data.columns[i]
        da1 = target_data.iloc[:, i].dropna().values

        value = list(target_data.iloc[:, i].dropna().values)
        thres1_right_list = [get_ISC_thres(value, tail='right', k=k) for k in k2p_list]
        thres1_left_list = [get_ISC_thres(value, tail='left', k=k) for k in k2p_list]
        
        count = st.count_list(da1)
        if np.max(count.iloc[:, 1] / len(da1)) > 0.1:
            logging += [[protein_name, col1, '', 'data predominant']]
            continue
        
        for j in range(target_number):
            col2 = target_data.columns[-(j + 1)]
            
            # get thres
            value = list(target_data.iloc[:, -(j + 1)].dropna().values)
            thres2_list = [get_ISC_thres(value, tail='right', k=k) for k in k2p_list]
            
            # cat data
            target_da = target_data.iloc[:, [i, -(j + 1)]].copy().dropna()
            length = target_da.shape[0]
            N = length
            if not length:
                logging += [[protein_name, col1, col2, 'empty target_da']]
                continue
                
            value_max = np.max(target_da.loc[:, col2])
            value_99 = np.quantile(target_da.loc[:, col2], 0.99)
            
            corr = target_da.corr().iloc[0, 1]
            if np.isnan(corr):
                logging += [[protein_name, col1, col2, 'corr na']]
                continue

            for pre_neg in [False, True]:
                direction = 'pos' if not pre_neg else 'neg'
                thres1_list = thres1_right_list if not pre_neg else thres1_left_list

                for a1, thres1 in enumerate(thres1_list):
                    thres1_backup = None
                    if not pre_neg:
                        predict_pos_set = set(target_da.loc[target_da.loc[:, col1] > thres1].index)
                        M = len(predict_pos_set)
                        if not M:
                            alpha_subs = alpha_list[a1]
                            thres1 = np.quantile(target_da.loc[:, col1], 1 - alpha_subs)
                            predict_pos_set = set(target_da.loc[target_da.loc[:, col1] > thres1].index)
                            logging += [[protein_name, col1, col2, '-'.join([direction, str(alpha_list[a1]), 'predict', 'None'])]]
                            thres1_backup = 'p subs'
                    else:
                        predict_pos_set = set(target_da.loc[target_da.loc[:, col1] < thres1].index)
                        M = len(predict_pos_set)
                        if not M:
                            alpha_subs = alpha_list[a1]
                            thres1 = np.quantile(target_da.loc[:, col1], alpha_subs)
                            predict_pos_set = set(target_da.loc[target_da.loc[:, col1] < thres1].index)
                            
                            logging += [[protein_name, col1, col2, '-'.join([direction, str(alpha_list[a1]), 'predict', 'None'])]]
                            thres1_backup = 'p subs'

                    M = len(predict_pos_set)
                    
                    select_values = target_da.loc[list(predict_pos_set), col2]
                    select_max = np.max(select_values)

                    for a2, thres2 in enumerate(thres2_list):
                        thres2_backup = None
                        exp_pos_set = set(target_da.loc[target_da.loc[:, col2] > thres2].index)
                        n = len(exp_pos_set)
                        if not n:
                            alpha_subs = alpha_list[a2]
                            thres2 = np.quantile(target_da.loc[:, col2], 1 - alpha_subs)
                            exp_pos_set = set(target_da.loc[target_da.loc[:, col2] > thres2].index)
                            
                            logging += [[protein_name, col1, col2, '-'.join([direction, str(alpha_list[a2]), 'experiment', 'None'])]]
                            thres2_backup = 'p subs'
                            n = len(exp_pos_set)

                        overlap_ind = list(predict_pos_set & exp_pos_set)
                        ratio = len(overlap_ind) / len(predict_pos_set)
                        percentage = len(exp_pos_set) / target_da.shape[0]

                        m = len(overlap_ind)
                        p_value = 1 - stats.hypergeom.cdf(m, N, M, n)
                    
                        result_all += [[protein_name, col1, col2, 
                                        alpha_list[a1], alpha_list[a2], thres1, thres2, value_max, value_99, select_max, 
                                        percentage, ratio, N, n, M, m, thres1_backup, thres2_backup,
                                        direction, p_value]]
                    
result_all_df = pd.DataFrame(result_all, columns=['Protein', 'property', 'DCM_col', 
                                                  'alpha1', 'alpha2', 'thres1', 'thres2', 'value_max', 'value_99', 'select_max', 
                                                  'percentage', 'overlap_ratio', 'N', 'n', 'M', 'm', 'thres1_backup', 'thres2_backup', 
                                                  'direction', 'p_value'])
result_all_df.loc[:, 'FC'] = result_all_df.loc[:, 'overlap_ratio'] / result_all_df.loc[:, 'percentage']
result_all_df.loc[:, 'log2FC'] = np.log2(result_all_df.loc[:, 'FC'])

logging_df = pd.DataFrame(logging, columns=['Protein', 'property', 'DCM_col', 'reason'])

result_all_df_use = result_all_df.loc[(result_all_df.loc[:, 'overlap_ratio'] != 0) & (result_all_df.loc[:, 'n'] != 0)]
min_non_zero_p = result_all_df_use.loc[result_all_df_use.loc[:, 'p_value'] > 0, 'p_value'].min()
result_all_df_use.loc[result_all_df_use.loc[:, 'p_value'] == 0, 'p_value'] = min_non_zero_p * 0.1
result_all_df_same_alpha = result_all_df_use.loc[result_all_df_use.loc[:, 'alpha1'] == result_all_df_use.loc[:, 'alpha2']]

# ana

In [13]:
result_alpha_dict = {}
for alpha in alpha_list:
    da_use = result_all_df_same_alpha.loc[result_all_df_same_alpha.loc[:, 'alpha1'] == alpha]
    best_index = [da.sort_values(['FC'], ascending=[False]).index[0] for _, da in da_use.groupby(['Protein'])]
    result_alpha_dict[alpha] = da_use.loc[best_index]

### 0.99分位数

In [14]:
follow_df = result_alpha_dict[0.001]
follow_index = follow_df.set_index(['Protein', 'property', 'DCM_col', 'direction']).index

In [15]:
result_alpha_dict_value = {}
for alpha in alpha_list:
    da_use = result_all_df_same_alpha.loc[result_all_df_same_alpha.loc[:, 'alpha1'] == alpha]
    best_index = [da.sort_values(['FC'], ascending=[False]).index[0] for _, da in da_use.groupby(['Protein'])]
    inter_data = da_use.loc[best_index].set_index(['Protein'])
    follow_use = da_use.set_index(['Protein', 'property', 'DCM_col', 'direction'])
    overlap = list(set(follow_index) & set(follow_use.index))
    follow_use = follow_use.loc[overlap].reset_index().set_index(['Protein'])
    inter_data.loc[follow_use.index] = follow_use
    result_alpha_dict_value[alpha] = inter_data

st.write_file('D:/设计/protein_evaluation/tables/DPF_enrich_DMS_0.001.xlsx', result_alpha_dict_value[0.001])

In [16]:
value_99_list = []
value_max_list = []
for alpha in alpha_list:
    plot_data = result_alpha_dict_value[alpha]  # .set_index('Protein')
    value_data = plot_data.loc[:, ['value_max', 'value_99', 'select_max']].copy()
    
    value_99 = (value_data.T / value_data.loc[:, 'value_99']).T.iloc[:, [-1]]
    value_max = (value_data.T / value_data.loc[:, 'value_max']).T.iloc[:, [-1]]
    value_99.columns = [alpha]
    value_max.columns = [alpha]
    
    value_99_list += [value_99]
    value_max_list += [value_max]
    
value_99_df = pd.concat(value_99_list, axis=1)
value_max_df = pd.concat(value_max_list, axis=1)

# bar plot

In [17]:
for alpha in alpha_list:
    plot_data = result_alpha_dict[alpha].set_index('Protein')
    
    plt.figure(figsize=(8, 4))
    for i, protein in enumerate(protein_list):
        if protein not in plot_data.index:
            continue

        h1 = plot_data.loc[protein, 'percentage']
        h2 = plot_data.loc[protein, 'overlap_ratio']
        max_h = np.max([h1, h2])

        p_value = plot_data.loc[protein, 'p_value']
        if p_value >= 0.0001:
            p_str = st.exactly_round(p_value, round_num_=4)
        else:
            p_str = f"{p_value:.2e}"

        plt.bar(i, h1, width=0.25, color='gray', label='Background')
        plt.bar(i + 0.3, h2, width=0.25, color=DC.c_purple, label='DPF')

        plt.text(i + 0.3, max_h + 0.01, p_str, ha='center', va='bottom')
        
        if not i:
            plt.legend()

    plt.xticks([i + 0.15 for i in range(len(protein_list))], protein_list, rotation=45, ha='right')
    plt.ylabel('Fraction positive (%)')
    ax = st.set_spines(plt)
    
    plt.savefig(save_path + 'barplot_fraction_' + str(alpha) + '.png', dpi=600, bbox_inches='tight', transparent=True)
    plt.close()
    
    plt.figure(figsize=(8, 4))
    for i, protein in enumerate(protein_list):
        if protein not in plot_data.index:
            continue
        if 'P53' not in protein and 'MAPK1' not in protein:
            continue

        h1 = plot_data.loc[protein, 'percentage']
        h2 = plot_data.loc[protein, 'overlap_ratio']
        max_h = np.max([h1, h2])

        p_value = plot_data.loc[protein, 'p_value']
        if p_value >= 0.0001:
            p_str = st.exactly_round(p_value, round_num_=4)
        else:
            p_str = f"{p_value:.2e}"

        plt.bar(i, h1, width=0.25, color='gray', label='Background')
        plt.bar(i + 0.3, h2, width=0.25, color=DC.c_purple, label='DPF')

        plt.text(i + 0.3, max_h * 1.05, p_str, ha='center', va='bottom')

    plt.xticks([i + 0.15 for i in range(len(protein_list))], [i if ('P53' in i) or ('MAPK1' in i) else '' for i in protein_list], rotation=45, ha='right')
    plt.ylabel('Fraction positive (%)')
    ax = st.set_spines(plt)
    
    plt.savefig(save_path + 'barplot_fraction_' + str(alpha) + '_P53.png', dpi=600, bbox_inches='tight', transparent=True)
    plt.close()

# compare with other methods

In [18]:
def create_broken_axis_plot(figsize=(6, 4), y11=0, y12=0.05, y21=0.7, y22=1, w1=None, break_height=0.05, 
                            d=0.015, break_left=True, break_right=True):

    # 创建图形和主坐标轴
    if w1 is None:
        d1 = y12 - y11
        d2 = y22 - y21
        w1 = d1 / (d1 + d2)
    w2 = 1 - w1  # w2 上半部分
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, 
                                   gridspec_kw={'height_ratios': [w2, w1]})
    # 设置共享x轴
    ax2.sharex(ax1)
    
    # 隐藏上图的x轴和下图的顶部边框
    ax1.spines['bottom'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.xaxis.set_visible(False)
    
    # 设置y轴范围
    ax1.set_ylim(y21, y22)
    ax2.set_ylim(y11, y12)
    
    # 绘制断裂符号
    w2_r = w2 / w1
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
    if break_left:
        ax1.plot((-d, +d), (- d, + d), **kwargs)        # 左上斜线
    if break_right:
        ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # 右上斜线
    
    kwargs.update(transform=ax2.transAxes)
    if break_left:
        ax2.plot((-d, +d), (1 - d * w2_r, 1 + d * w2_r), **kwargs)  # 左下斜线
    if break_right:
        ax2.plot((1 - d, 1 + d), (1 - d * w2_r, 1 + d * w2_r), **kwargs)  # 右下斜线
    
    return fig, ax1, ax2

In [19]:
methods_list = []
methods_alpha_dict = {}
corr_info = []
for alpha in alpha_list:
    data_compare = result_alpha_dict[alpha]

    res1_list = []
    res2_list = []
    for ind in data_compare.index:
        protein = data_compare.loc[ind, 'Protein']
        if protein in 'PafA':
            continue
        
        my_col = data_compare.loc[ind, 'property']
        DCM_col = data_compare.loc[ind, 'DCM_col']
        DCM_thres = data_compare.loc[ind, 'thres2']
        choose = data_compare.loc[ind, 'M']
        my_m = data_compare.loc[ind, 'm']

        file = protein_name_dict[protein]
        pre_name = file_name_dict[file]

        data = st.read_file(path + file, index_col=0)
        data_add = st.read_file(add_path + protein + '.txt', index_col=0)
        data = pd.concat([data_add, data], axis=1)
        
        methods_columns = [i for i in data.columns if 'DMS' not in i]
        target_columns = [i for i in data.columns if 'DMS' in i]
        target_number = len(target_columns)

        data_pre = st.read_file(predict_path + pre_name, index_col=0)
        pre_col_use = [i for i in data_pre.columns if '.1' not in i]
        data_pre = data_pre.loc[:, pre_col_use]
        
        # orthogonal start
        if alpha == 0.1:
            trans = True if np.mean(data_pre.loc[:, my_col]) > 0.9 else False
            orthogonal_df = pd.concat([data_add, data_pre.loc[:, [my_col]]], axis=1).dropna()
            corr1 = orthogonal_df.corr().iloc[0, 1]
            corr2 = pd.concat([np.log10(data_add), np.log10(data_pre.loc[:, [my_col]])], axis=1).dropna().corr().iloc[0, 1]
            corr_info += [[protein, corr1, corr2]]
            label_info = 'corr:' + str(round(corr1, 4))

            plt.figure(figsize=(4, 4))
            if not trans:
                plt.scatter(orthogonal_df.iloc[:, 0], orthogonal_df.iloc[:, 1], s=1, label=label_info)
            else:
                plt.scatter(orthogonal_df.iloc[:, 0], 1 - orthogonal_df.iloc[:, 1], s=1, label=label_info)
            plt.xscale('log')
            plt.yscale('log')
            plt.title(protein)
            plt.legend()
            plt.savefig(save_path + 'corr/corr_' + protein + '.png', dpi=600, bbox_inches='tight', transparent=True)
            plt.close()
        # orthogonal end
        
        index_use = pd.concat([data_pre.loc[:, [my_col]], data.loc[:, DCM_col]], axis=1).dropna().index
        data_use = data.loc[index_use]
        
        res1 = [choose, my_m]
        for col in methods_columns:
            da = data.sort_values(col, ascending=False).loc[:, col].iloc[:choose]
            if da.iloc[0] == da.iloc[-1]:
                same_value = da.iloc[0]
                exp_value = np.mean(data.loc[data.loc[:, col] == same_value, DCM_col] > DCM_thres) * choose
                res1 += [exp_value]
            else:
                res1 += [np.sum(data.sort_values(col, ascending=False).loc[:, DCM_col].iloc[:choose] > DCM_thres)]
                
        res1 = [res1]
        res1 = pd.DataFrame(res1, index=[protein], columns=['choose', 'DPF'] + methods_columns)
        res1_list += [res1]

    res1_df = pd.concat(res1_list)
    res1_df_clean = res1_df.T.dropna().T

    methods_alpha_dict[alpha] = (res1_df, data_compare.set_index('Protein').loc[:, ['percentage']])
    methods_list += [list(res1_df.columns[(np.mean(res1_df == res1_df, axis=0) > 0.4)])]

columns_use = list(set(methods_list[0]) & set(methods_list[1]) & set(methods_list[2]) & set(methods_list[3]))
columns_use = [i for i in res1_df.columns if i in columns_use]

In [20]:
for alpha in alpha_list:
    res1_df_raw, background = methods_alpha_dict[alpha]
    res1_df = res1_df_raw.copy()
    res1_df = res1_df.loc[:, columns_use]
    index_sort = [i for i in protein_list if i in res1_df.index]
    res1_df = res1_df.loc[index_sort]

    rank_df1 = rank_rows_descending_with_ties_int(res1_df)
    mean_rank = pd.DataFrame(np.mean(rank_df1.iloc[:, 1:], axis=0).sort_values())
    fraction_df = res1_df.iloc[:, 1:].T / res1_df.iloc[:, 0].T
    
#     break

pvalue_df = res1_df.iloc[:, 1:].copy().astype(float)
meta_da = result_alpha_dict[alpha].set_index('Protein').loc[res1_df.index, ['N', 'n', 'M']]

for protein in meta_da.index:
    N = meta_da.loc[protein, 'N']
    n = meta_da.loc[protein, 'n']
    M = meta_da.loc[protein, 'M']
    
    for col in pvalue_df.columns:
        m = res1_df.loc[protein, col]
        m = int(np.round(m, 0))
        
        p_value = 1 - stats.hypergeom.cdf(m, N, M, n)
        pvalue_df.loc[protein, col] = p_value

# ','.join(fraction_df.index)

pvalue_df_fill_0 = pvalue_df.copy()
min_pvalue = np.min(pvalue_df[pvalue_df > 0])
pvalue_df_fill_0[pvalue_df_fill_0 == 0] = min_pvalue

performance_df = pd.DataFrame([np.mean(1 - pvalue_df, axis=0), 
                               np.mean(- np.log10(pvalue_df_fill_0), axis=0), 
                               np.mean(fraction_df, axis=1), 
                               mean_rank.iloc[:, 0]], 
                              index=['1-P', '-log10(P)', 'Fraction', 'Rank']).T

In [22]:
colors = [
    '#FF0000', '#B3FFB3', '#B3B3FF', '#B3FFFF',
    '#FFFFB3', '#FFE0B3', '#E0B3E0', '#B3E0B3',
    '#B3B3E0', '#FFD9CC', '#E0E0B3', '#F0E6D2',
    '#C6D9EC', '#E0C6B3'
]

colors = [
    '#FF0000', '#FF82AB', '#B3FFB3', '#B3B3FF', '#B3FFFF',
    '#FFFFB3', '#FFE0B3', '#E0B3E0', '#B3E0B3',
    '#B3B3E0', '#FFD9CC', '#E0E0B3', '#F0E6D2',
    '#C6D9EC', '#E0C6B3'
]

color_dict = {method: color for method, color in zip(columns_use[1:], colors)}

In [23]:
# 绘图设置
bar_width = 0.8
globle_min = 0.05

for col in performance_df.columns:
    min_val = np.min(performance_df.loc[performance_df.loc[:, col] > 0, col])
    max_val = np.max(performance_df.loc[:, col])
    
    change_fold = 10 ** np.ceil(np.log10(1 / min_val))
    
    threshold = np.floor((min_val) * change_fold) / change_fold
#     threshold = max(threshold, globle_min)
    threshold_upper = np.ceil(max_val * 10) / 10
    
    y12 = globle_min if globle_min < threshold else threshold / 2

    fig, ax1, ax2 = create_broken_axis_plot(figsize=(10, 3), 
                                            y11=0, y12=y12, y21=threshold, y22=threshold_upper, 
                                            w1=0.1, d=0.005, break_left=True, break_right=False)

    # 绘制分组条形图
    for i, ind in enumerate(performance_df.index):
        value = performance_df.loc[ind, col]
        ax1.bar(i, value, width=bar_width, color=color_dict[ind], alpha=0.7)
        ax2.bar(i, min(globle_min, value), width=bar_width, color=color_dict[ind], alpha=0.7)

    ax2.set_xticks(range(performance_df.shape[0]), list(performance_df.index), rotation=30, ha='right')
    ax2.set_ylim(0)

    ax1.set_title(col)

    plt.subplots_adjust(hspace=0.1)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.savefig('D:/设计/protein_evaluation/revision/compare_DMS_more_metrics/metrics_' + col + '_0.001.png', 
                dpi=600, bbox_inches='tight', transparent=True)
#     plt.show()
    plt.close()

In [24]:
for alpha in alpha_list:
    res1_df_raw, background = methods_alpha_dict[alpha]
    res1_df = res1_df_raw.copy()
    res1_df = res1_df.loc[:, columns_use]
    index_sort = [i for i in protein_list if i in res1_df.index]
    res1_df = res1_df.loc[index_sort]

    rank_df1 = rank_rows_descending_with_ties_int(res1_df)
    mean_rank = pd.DataFrame(np.mean(rank_df1.iloc[:, 1:], axis=0).sort_values())
    fraction_df = res1_df.iloc[:, 1:].T / res1_df.iloc[:, 0].T

    color_list = [color_dict[ind] for ind in fraction_df.index]
    plt.figure(figsize=(8, 6))

    # 为每列设置基础横坐标（列索引）
    base_x_positions = np.arange(fraction_df.shape[1])

    # 为每个数据点生成随机横坐标偏移
    np.random.seed(42)
    x_offsets = np.random.normal(0, 0.1, fraction_df.shape)
    x_coords = base_x_positions + x_offsets

    # 绘制每个数据点
    for ind_idx in range(fraction_df.shape[0]):
        y_values = fraction_df.iloc[ind_idx, :].values
        x_values = x_coords[ind_idx, :]

        alpha_plot = [0.7 if not ind_idx else 1] * 14

        plt.scatter(x_values, y_values, 
                   color=color_list[ind_idx], 
                   alpha=alpha_plot, 
                   s=80,
                   label=fraction_df.index[ind_idx], zorder=20 - ind_idx)

    for i, col in enumerate(fraction_df.columns):
        back_val = background.loc[col, 'percentage']
        plt.plot([i - 0.4, i + 0.4], [back_val, back_val], lw=1, color='black', alpha=0.7, ls='--')


    # 设置图形属性
    plt.ylabel('Fraction positive (%)', fontsize=15)
    # plt.title('Data Values with Horizontal Dispersion')
    plt.xticks(range(fraction_df.shape[1]), fraction_df.columns, 
               rotation=30, ha='right')
    plt.ylim(-0.1, 1.1)  # 稍微扩展y轴范围以便更好地显示边界点
    st.set_spines(plt)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.savefig(save_path + 'scatter_' + str(alpha) + '.png', dpi=600, bbox_inches='tight', transparent=True)
#     plt.close()

    plt.figure(figsize=(2.5, 4))

    x1 = 0
    x2 = 2

    plt.text(x1, 0, 'Method', ha='left', va='center')
    plt.text(x2, 0, 'Mean rank', ha='left', va='center')
    for i, ind in enumerate(mean_rank.index):
        plt.scatter(x1, - i - 1, c=color_dict[ind], s=100)

        fc = DC.c_red if ind in 'DPF' else 'black'
        plt.text(x1 + 0.2, - i - 1, ind, ha='left', va='center', color=fc)
        plt.text(x2, - i - 1, st.exactly_round(mean_rank.loc[ind, 0], round_num_=2), ha='left', va='center', color=fc)

    plt.arrow(x2 + 0.8, - i - 1, 0, 13.5, 
              head_width=0.1, head_length=0.5, 
              fc='black', ec='black', 
              length_includes_head=True)

    plt.xlim(x1-0.1, x2 + 1)
    plt.axis('off')
    
    plt.savefig(save_path + 'scatter_' + str(alpha) + '_rank_add_esm.png', dpi=600, bbox_inches='tight', transparent=True)
#     plt.show()
    plt.close()

In [25]:
st.write_file('D:/设计/protein_evaluation/tables/compare_DMS_ranking_0.001.xlsx', rank_df1)
st.write_file('D:/设计/protein_evaluation/tables/compare_DMS_Hit_0.001.xlsx', res1_df)